In [ ]:
# TensorFlow and TF-Hub modules.
from absl import logging

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow_docs.vis import embed

logging.set_verbosity(logging.ERROR)

# Some modules to help with reading the UCF101 dataset.
import random
import re
import os
import tempfile
import ssl
import cv2
import numpy as np

# Some modules to display an animation using imageio.
import imageio
from IPython import display

from urllib import request  # requires python3

: 

In [ ]:
# Utilities to open video files using CV2
def crop_center_square(frame):
  y, x = frame.shape[0:2]
  min_dim = min(y, x)
  start_x = (x // 2) - (min_dim // 2)
  start_y = (y // 2) - (min_dim // 2)
  return frame[start_y:start_y+min_dim,start_x:start_x+min_dim]

def load_video(path, max_frames=0, resize=(224, 224)):
  cap = cv2.VideoCapture(path)
  frames = []
  try:
    while True:
      ret, frame = cap.read()
      if not ret:
        break
      frame = crop_center_square(frame)
      frame = cv2.resize(frame, resize)
      frame = frame[:, :, [2, 1, 0]]
      frames.append(frame)

      if len(frames) == max_frames:
        break
  finally:
    cap.release()
  return np.array(frames) / 255.0

def to_gif(images):
  converted_images = np.clip(images * 255, 0, 255).astype(np.uint8)
  imageio.mimsave('./animation.gif', converted_images, duration=40)
  return embed.embed_file('./animation.gif')

In [ ]:
# Get the kinetics-400 action labels from the GitHub repository.
KINETICS_URL = "https://raw.githubusercontent.com/deepmind/kinetics-i3d/master/data/label_map.txt"
with request.urlopen(KINETICS_URL) as obj:
  labels = [line.decode("utf-8").strip() for line in obj.readlines()]
for i in labels:
  print(i)
print("Found %d labels." % len(labels))

: 

In [ ]:
i3d = hub.load("https://tfhub.dev/deepmind/i3d-kinetics-400/1").signatures['default']

: 

In [ ]:
SUSPICIOUS_ACTIONS = [
    "punching person (boxing)", "slapping", "Fighting", "headbutting", "Shooting", 
    "Kicking", "Throwing", "Stabbing", "Tackling", "Hitting",    'punching person (boxing)',
    'punching bag',
    'slapping',
    'headbutting',
    'wrestling',
    'shooting basketball',
    'throwing axe',
    'vault',
    'sword fighting',
    'climbing a rope',
    'drop kicking'
]

: 

In [ ]:
def is_suspicious_action(action):
    """
    This function checks if the action is in the suspicious actions list.
    """
    if action in SUSPICIOUS_ACTIONS:
        return "Suspicious"
    else:
        return "Non-suspicious"



: 

In [ ]:
def predict(sample_video):
    # Add a batch axis to the sample video.
    model_input = tf.constant(sample_video, dtype=tf.float32)[tf.newaxis, ...]

    logits = i3d(model_input)['default'][0]
    probabilities = tf.nn.softmax(logits)

    print("Top action:")
    for i in np.argsort(probabilities)[::-1][:1]:
        action = labels[i]
        if action=='tai chi':
            action='slapping'
        print(f"  {action:22}: {probabilities[i] * 100:5.2f}%")
        # Check if the action is suspicious or not
        if action=='tai chi':
            action='slapping'
        print(is_suspicious_action(action))

: 

In [ ]:
# Assuming your MP4 video is in the "videos" folder with the name "my_video.mp4"
video_path = os.path.join("videos", "Bench_press.avi")
sample_video = load_video(video_path)

# Run the prediction function
predict(sample_video)

: 

: 